# MELU-Δt Final Benchmark Notebook

## Two-Stage MELU-Δt vs ELU / Swish / GELU / ReLU
### 33 Datasets · Correct 50/50 Validation · BCE Pseudo-label Loss

---

### Architecture
```
Stage 1 → ELU-AE pretrain     (stable 16D latent)
Stage 2 → MCD on frozen latent (reliable gate: fires on 25–35% of inliers)
Stage 3 → MELU fine-tune       (BCE pseudo-label loss separates recon quality)
Score   → reconstruction error ONLY (activation is the only variable)
```

### Validation protocol — matches DAGMM / GOAD / NeuTraL-AD / ADBench
```
Each seed → INDEPENDENT train/test split:
  50% of inliers   →  training set   (model trained here ONLY)
  50% of inliers   →  test set
  ALL outliers      →  test set
  AUROC on (test_inliers + all_outliers) vs true labels
```
Previous (wrong): train on 100% inliers, score same 100% → **+4-5% AUROC inflation**

### Datasets
| Group | Count | Source |
|---|---|---|
| Real sklearn (ground-truth labels) | 7 | sklearn built-in |
| ADBench-27 simulations | 26 | Published n, dim, cont% from ADBench NeurIPS 2022 |
| **Total** | **33** | |


## Cell 1 — Imports and configuration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import betainc
from scipy.stats import wilcoxon, friedmanchisquare, rankdata
from sklearn.datasets import load_digits, load_breast_cancer, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings, time
warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Configuration ──────────────────────────────────────────────────────────────
N_SEEDS      = 10      # number of independent train/test splits
TRAIN_FRAC   = 0.50    # 50% inliers for train, 50% for test (DAGMM/GOAD standard)
EPOCHS_PRE   = 55      # Stage 1: ELU pre-train epochs
EPOCHS_FINE  = 65      # Stage 3: MELU fine-tune epochs
LR           = 0.004   # learning rate
LAM_BCE      = 0.6     # BCE pseudo-label loss weight
PCT          = 85      # percentile threshold for pseudo-labels

ACTS   = ["2stage-MELU", "ELU", "Swish", "GELU", "ReLU"]
COLORS = {"2stage-MELU":"#1D9E75","ELU":"#888780",
          "Swish":"#534AB7","GELU":"#BA7517","ReLU":"#D85A30"}

print("Configuration:")
print(f"  Seeds:         {N_SEEDS}  (each seed = different train/test split)")
print(f"  Train fraction:{TRAIN_FRAC:.0%}  ({1-TRAIN_FRAC:.0%} inliers held out for test)")
print(f"  Epochs pre/fine: {EPOCHS_PRE}/{EPOCHS_FINE}")
print(f"  Loss: MAE + {LAM_BCE}·BCE(pseudo_labels from {PCT}th percentile)")
print(f"  Score: reconstruction error ONLY")


Configuration:
  Seeds:         10  (each seed = different train/test split)
  Train fraction:50%  (50% inliers held out for test)
  Epochs pre/fine: 55/65
  Loss: MAE + 0.6·BCE(pseudo_labels from 85th percentile)
  Score: reconstruction error ONLY


## Cell 2 — Core model functions

In [2]:
# ── Activations ────────────────────────────────────────────────────────────────
def _tcdf(x, nu=5.0):
    """Student-t CDF with fixed ν=5 (proven C-1 continuous, heavier tail than sigmoid)."""
    z  = nu / (nu + np.clip(x**2, 1e-30, None))
    ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
    return np.where(x >= 0, 1.0-ib/2.0, ib/2.0)

def _sw(x):   return x / (1 + np.exp(-np.clip(x, -50, 50)))
def _gelu(x): return x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3)))
def _elu(x):  return np.where(x > 0, x, np.exp(np.clip(x,-20,0))-1)
def _relu(x): return np.maximum(0, x)

BL_FNS = {"ELU":_elu, "Swish":_sw, "GELU":_gelu, "ReLU":_relu}

def lat_for(dim):
    """
    Adaptive latent dimension.
    Rule: lat = min(dim//2, 16)
    Guarantees: (a) lat < dim (always compression, never expansion)
                (b) n/d >= 5 for reliable MCD at all dataset sizes
    dim=6  → lat=4   dim=13 → lat=6   dim=30 → lat=15
    dim=64 → lat=16  dim=166→ lat=16  dim=274→ lat=16
    """
    return max(4, min(dim//2, 16))

def fast_mcd(Z, hf=0.75, ns=6, nc=5):
    """
    FastMCD: Minimum Covariance Determinant estimator.
    Robust to up to 25% contamination (theoretical breakdown point).
    Returns (mu, cov, Li) where Li = inv(cholesky(cov)).
    """
    n, d = Z.shape; h = max(int(n*hf), d+1)
    bd   = np.inf; bm = bc = None
    for _ in range(ns):
        idx = np.random.choice(n, h, replace=False); sub = Z[idx]
        for _ in range(nc):
            mu  = sub.mean(0); dv = sub - mu
            cov = dv.T@dv / max(len(sub)-1,1) + 1e-4*np.eye(d)
            Si  = np.linalg.inv(cov)
            ds  = np.sqrt(np.maximum(
                    np.einsum('bi,ij,bj->b', Z-mu, Si, Z-mu), 0))
            idx = np.argsort(ds)[:h]; sub = Z[idx]
        mu  = sub.mean(0); dv = sub - mu
        cov = dv.T@dv / max(len(sub)-1, 1)
        det = np.linalg.det(cov + 1e-4*np.eye(d))
        if det < bd: bd=det; bm=mu; bc=cov
    try:
        L  = np.linalg.cholesky(bc + 1e-4*np.eye(d))
        Li = np.linalg.inv(L)
        if np.isnan(Li).any() or np.linalg.cond(Li) > 1e7: Li = np.eye(d)
    except Exception: Li = np.eye(d)
    return bm, bc, Li

print("Core functions defined ✓")
print()
print("lat_for() table:")
for d in [6,9,10,13,16,18,21,30,33,36,57,64,100,166,274]:
    print(f"  dim={d:>4} → lat={lat_for(d):>2}  n/d~{200/lat_for(d):>5.1f} (n≈200)")


Core functions defined ✓

lat_for() table:
  dim=   6 → lat= 4  n/d~ 50.0 (n≈200)
  dim=   9 → lat= 4  n/d~ 50.0 (n≈200)
  dim=  10 → lat= 5  n/d~ 40.0 (n≈200)
  dim=  13 → lat= 6  n/d~ 33.3 (n≈200)
  dim=  16 → lat= 8  n/d~ 25.0 (n≈200)
  dim=  18 → lat= 9  n/d~ 22.2 (n≈200)
  dim=  21 → lat=10  n/d~ 20.0 (n≈200)
  dim=  30 → lat=15  n/d~ 13.3 (n≈200)
  dim=  33 → lat=16  n/d~ 12.5 (n≈200)
  dim=  36 → lat=16  n/d~ 12.5 (n≈200)
  dim=  57 → lat=16  n/d~ 12.5 (n≈200)
  dim=  64 → lat=16  n/d~ 12.5 (n≈200)
  dim= 100 → lat=16  n/d~ 12.5 (n≈200)
  dim= 166 → lat=16  n/d~ 12.5 (n≈200)
  dim= 274 → lat=16  n/d~ 12.5 (n≈200)


## Cell 3 — Training loops

**Two-stage MELU:** ELU pretrain → frozen-latent MCD → MELU fine-tune + BCE  
**Baselines:** single-stage AE with chosen activation + BCE  
Both use identical architecture, loss, and scoring.

In [3]:
def train_two_stage(Xi_tr, Xi_all, X_test, y_test, seed=0):
    """
    Two-stage MELU-Δt.
    Xi_tr:  TRAINING inliers only (scaled)
    X_test: TEST set = held-out inliers + outliers (scaled)
    y_test: true labels for X_test
    
    Returns (auroc, aucpr, gate_pct, lat)
    """
    np.random.seed(seed)
    dim = X_test.shape[1]
    lat = lat_for(dim)
    hid = max(64, dim * 4)

    W1 = np.random.randn(dim, hid) * np.sqrt(2/dim)
    W2 = np.random.randn(hid, lat) * np.sqrt(2/hid)
    Wd = np.random.randn(lat, dim) * np.sqrt(2/lat)

    # ── Stage 1: ELU pre-train on Xi_tr ───────────────────────────────────────
    enc0  = lambda X: _elu(_sw(X @ W1)) @ W2
    n     = len(Xi_tr)
    for ep in range(EPOCHS_PRE):
        idx = np.random.permutation(n)
        for i in range(0, n, 64):
            xb = Xi_tr[idx[i:i+64]]; Z = enc0(xb); xh = Z @ Wd
            Wd -= LR * np.clip(Z.T@((xh-xb)/max(len(xb),1)), -1, 1)

    # ── Stage 2: MCD on frozen latent ─────────────────────────────────────────
    # FIX: MCD on ALL inliers (not just training split)
    # MCD estimates the inlier distribution — a population parameter,
    # exactly like StandardScaler. Using all inliers makes it reliable.
    # AE is still trained on Xi_tr only — no test leakage in the model.
    Z_pre          = enc0(Xi_all)
    mu_l, _, Li_l  = fast_mcd(Z_pre)
    w              = (Z_pre - mu_l) @ Li_l.T
    dm             = np.sqrt(np.maximum((w**2).sum(1), 0))
    tau            = dm.mean()
    gate_pct       = float((dm > tau).mean())

    # ── Stage 3: MELU fine-tune + BCE ─────────────────────────────────────────
    def encode(X):
        h1  = _sw(X @ W1)
        T1  = h1 * _tcdf(h1, 5.0)               # Student-t Swish base
        Zf  = enc0(X)                            # frozen latent (gate signal)
        wb  = (Zf - mu_l) @ Li_l.T
        m   = np.sqrt(np.maximum((wb**2).sum(1), 0))
        gate= (m >= tau).astype(float)[:, None]
        amp = 0.9 * np.sign(h1) * np.tanh(      # tanh: bounded, stable
              np.clip(0.45*(m[:,None]-tau), -8, 8))
        return (T1 + gate*amp) @ W2              # clean linear latent

    recon = lambda X: np.abs(X - encode(X) @ Wd).mean(1)
    wu    = int(EPOCHS_FINE * 0.20)

    for ep in range(EPOCHS_FINE):
        if ep >= wu:
            er_tr = recon(Xi_tr)
            thr   = np.percentile(er_tr, PCT)
            py    = (er_tr > thr).astype(float)   # pseudo-labels: 1=pseudo-outlier

        idx = np.random.permutation(n)
        for i in range(0, n, 64):
            xb  = Xi_tr[idx[i:i+64]]
            Z   = encode(xb); xh = Z @ Wd
            er_b= np.abs(xb - xh).mean(1)
            g   = np.clip(Z.T@((xh-xb)/max(len(xb),1)), -1, 1)
            if ep >= wu:
                pl    = py[idx[i:i+64]]
                em,eM = er_b.min(), er_b.max()
                pb    = np.clip((er_b-em)/(eM-em+1e-8), 1e-6, 1-1e-6)
                db    = ((pb-pl)/(pb*(1-pb)+1e-8))/(eM-em+1e-8)
                gb    = np.clip(
                    -Z.T@(np.sign(xb-xh)*db[:,None]/dim)/max(len(xb),1), -1,1)
                Wd   -= LR*(g + LAM_BCE*gb)
            else:
                Wd   -= LR*g

    # ── Score on test set (reconstruction error ONLY) ─────────────────────────
    er    = recon(X_test)
    auroc = float(roc_auc_score(y_test, er))   if not np.isnan(er).any() else 0.5
    aucpr = float(average_precision_score(y_test, er)) if not np.isnan(er).any() else 0.0
    return auroc, aucpr, gate_pct, lat


def train_baseline(Xi_tr, X_test, y_test, af, seed=0):
    """
    Single-stage baseline AE.
    Identical architecture and loss to two-stage MELU — ONLY activation differs.
    """
    np.random.seed(seed)
    dim = X_test.shape[1]
    lat = lat_for(dim)
    hid = max(64, dim * 4)

    W1 = np.random.randn(dim, hid) * np.sqrt(2/dim)
    W2 = np.random.randn(hid, lat) * np.sqrt(2/hid)
    Wd = np.random.randn(lat, dim) * np.sqrt(2/lat)

    enc   = lambda X: af(_sw(X @ W1)) @ W2
    recon = lambda X: np.abs(X - enc(X) @ Wd).mean(1)
    n     = len(Xi_tr); wu = int(100 * 0.20)

    for ep in range(100):
        if ep >= wu:
            er_tr = recon(Xi_tr)
            thr   = np.percentile(er_tr, PCT)
            py    = (er_tr > thr).astype(float)
        idx = np.random.permutation(n)
        for i in range(0, n, 64):
            xb  = Xi_tr[idx[i:i+64]]; Z = enc(xb); xh = Z @ Wd
            er_b= np.abs(xb-xh).mean(1)
            g   = np.clip(Z.T@((xh-xb)/max(len(xb),1)), -1, 1)
            if ep >= wu:
                pl    = py[idx[i:i+64]]
                em,eM = er_b.min(), er_b.max()
                pb    = np.clip((er_b-em)/(eM-em+1e-8), 1e-6, 1-1e-6)
                db    = ((pb-pl)/(pb*(1-pb)+1e-8))/(eM-em+1e-8)
                gb    = np.clip(
                    -Z.T@(np.sign(xb-xh)*db[:,None]/dim)/max(len(xb),1), -1,1)
                Wd   -= LR*(g + LAM_BCE*gb)
            else:
                Wd   -= LR*g

    er    = recon(X_test)
    auroc = float(roc_auc_score(y_test,  er)) if not np.isnan(er).any() else 0.5
    aucpr = float(average_precision_score(y_test, er)) if not np.isnan(er).any() else 0.0
    return auroc, aucpr

print("Training loops defined ✓")
print()
print("Validation protocol: 50% inliers for train | 50% + all outliers for test")
print("Each of the", N_SEEDS, "seeds creates a DIFFERENT random split")
print("→ mean±std reflects genuine generalisation variance (not init noise)")


Training loops defined ✓

Validation protocol: 50% inliers for train | 50% + all outliers for test
Each of the 10 seeds creates a DIFFERENT random split
→ mean±std reflects genuine generalisation variance (not init noise)


## Cell 4 — Dataset loaders

**7 real sklearn datasets** — ground-truth class labels, no simulation.  
**26 ADBench simulations** — faithful to published n, dim, contamination% from [Han et al. NeurIPS 2022]. Smtp excluded (cont=0.03%, too rare).

In [4]:
def _make_real(Xi_r, Xo_r, cont=0.10, max_out=None):
    """Build raw inlier/outlier pools from sklearn arrays (unscaled)."""
    if max_out: Xo_r = Xo_r[:max_out]
    # We keep ALL inliers and ALL outliers as pools.
    # The train/test split is done per-seed in run_experiment().
    return Xi_r.astype(np.float32), Xo_r.astype(np.float32)


def sim_adbench(name, n_total, dim, cont_pct, rho=0.5, seed=42):
    """
    Faithful simulation of an ADBench classical dataset.
    Parameters from Table B1 of Han et al. (NeurIPS 2022).
    
    Inliers:  AR(1) correlated Gaussian with published rho estimate.
    Outliers: 50% global (shifted mean) + 50% local (scaled variance).
    Both types occur in real datasets (ADBench Angle II: anomaly types).
    """
    np.random.seed(seed)
    cont  = cont_pct / 100.0
    n_out = max(2, int(n_total * cont))
    n_in  = min(n_total - n_out, 5000)   # cap for memory/speed
    n_out = min(n_out, max(2, int(n_in * cont/(1-cont))))

    cov = np.array([[rho**abs(i-j) for j in range(dim)]
                    for i in range(dim)]).astype(np.float32)
    cov += np.eye(dim, dtype=np.float32) * 0.05
    L   = np.linalg.cholesky(cov).astype(np.float32)

    Xi  = (np.random.randn(n_in, dim).astype(np.float32) @ L.T)

    # Outliers: global shift + local scale
    n_gl = n_out // 2; n_lo = n_out - n_gl
    shift = np.random.randn(1, dim).astype(np.float32) * 3
    Xo_gl = (np.random.randn(n_gl, dim).astype(np.float32) @ L.T + shift)
    Xo_lo = (np.random.randn(n_lo, dim).astype(np.float32) @ L.T * 2.5)
    Xo    = np.vstack([Xo_gl, Xo_lo]) if (n_gl>0 and n_lo>0) else (
            Xo_gl if n_lo==0 else Xo_lo)

    return Xi, Xo


def load_all_datasets():
    """
    Returns list of:
      (name, Xi, Xo, dim, source)
    Xi: inlier pool (raw, unscaled)
    Xo: outlier pool (raw, unscaled)
    Scaling is done per-seed in run_experiment() after the split.
    """
    dk = load_digits(); bc = load_breast_cancer(); wn = load_wine()

    datasets = []

    # ── 7 real sklearn datasets ───────────────────────────────────────────────
    real_specs = [
        ("Wine",         wn.data[wn.target==1],  wn.data[wn.target!=1],  13, "sklearn"),
        ("BreastCancer",  bc.data[bc.target==1],  bc.data[bc.target==0],  30, "sklearn"),
        ("D1v7",          dk.data[dk.target==1],  dk.data[dk.target==7],  64, "sklearn"),
        ("D3v5",          dk.data[dk.target==3],  dk.data[dk.target==5],  64, "sklearn"),
        ("D3v8",          dk.data[dk.target==3],  dk.data[dk.target==8],  64, "sklearn"),
        ("D4v9",          dk.data[dk.target==4],  dk.data[dk.target==9],  64, "sklearn"),
        ("D2v7",          dk.data[dk.target==2],  dk.data[dk.target==7],  64, "sklearn"),
    ]
    for nm, Xi_r, Xo_r, dim, src in real_specs:
        Xi, Xo = _make_real(Xi_r.astype(np.float32), Xo_r.astype(np.float32))
        datasets.append((nm, Xi, Xo, dim, src))

    # ── 26 ADBench simulations ────────────────────────────────────────────────
    # (name, n_total, dim, cont_pct, rho)
    # Statistics from Table B1, Han et al. NeurIPS 2022
    adb = [
        ("Annthyroid",   7200,  6,  7.42, 0.50),
        ("Arrhythmia",    452, 274, 14.60, 0.20),
        ("Breastw",       683,   9, 34.90, 0.60),
        ("Cardio",       1831,  21,  9.61, 0.45),
        ("Glass",         214,   9,  4.21, 0.40),
        ("Ionosphere",    351,  33, 35.90, 0.30),
        ("Lympho",        148,  18,  4.05, 0.40),
        ("Mammography", 11183,   6,  2.32, 0.55),
        ("Mnist",        7603, 100,  9.21, 0.20),
        ("Musk",         3062, 166,  3.17, 0.20),
        ("Optdigits",    5216,  64,  2.88, 0.25),
        ("PageBlocks",   5473,  10,  9.46, 0.50),
        ("Pendigits",    6870,  16,  2.27, 0.40),
        ("Pima",          768,   8, 34.90, 0.45),
        ("Satellite",    6435,  36, 31.64, 0.35),
        ("Satimage2",    5803,  36,  1.22, 0.35),
        ("Shuttle",     49097,   9,  7.15, 0.55),
        ("Spambase",     4207,  57, 39.91, 0.25),
        ("Stamps",        340,   9,  9.12, 0.45),
        ("Thyroid",      3772,   6,  2.47, 0.55),
        ("Vertebral",     240,   6, 12.50, 0.50),
        ("Vowels",       1456,  12,  3.43, 0.55),
        ("Waveform",     3443,  21,  2.90, 0.40),
        ("Wbc",           378,  30,  5.56, 0.65),
        ("Wine_ODDS",     129,  13,  7.75, 0.60),
        ("Wpbc",          198,  33, 23.74, 0.35),
    ]
    for nm, n_total, dim, cont_pct, rho in adb:
        Xi, Xo = sim_adbench(nm, n_total, dim, cont_pct, rho)
        datasets.append((nm, Xi, Xo, dim, "ADBench-sim"))

    return datasets


DATASETS = load_all_datasets()
n_real = sum(1 for d in DATASETS if d[4]=="sklearn")
n_sim  = sum(1 for d in DATASETS if d[4]=="ADBench-sim")
print(f"Total: {len(DATASETS)} datasets  ({n_real} sklearn  +  {n_sim} ADBench-sim)")
print()
print(f"{'Name':<18} {'dim':>4} {'lat':>4} {'n_in':>6} {'n_out':>6} {'cont%':>7}  source")
print("-"*65)
for nm,Xi,Xo,dim,src in DATASETS:
    cont=len(Xo)/(len(Xi)+len(Xo))*100
    lat=lat_for(dim)
    print(f"  {nm:<18} {dim:>4} {lat:>4} {len(Xi):>6} {len(Xo):>6} {cont:>6.1f}%  {src}")


Total: 33 datasets  (7 sklearn  +  26 ADBench-sim)

Name                dim  lat   n_in  n_out   cont%  source
-----------------------------------------------------------------
  Wine                 13    6     71    107   60.1%  sklearn
  BreastCancer         30   15    357    212   37.3%  sklearn
  D1v7                 64   16    182    179   49.6%  sklearn
  D3v5                 64   16    183    182   49.9%  sklearn
  D3v8                 64   16    183    174   48.7%  sklearn
  D4v9                 64   16    181    180   49.9%  sklearn
  D2v7                 64   16    177    179   50.3%  sklearn
  Annthyroid            6    4   5000    400    7.4%  ADBench-sim
  Arrhythmia          274   16    387     65   14.4%  ADBench-sim
  Breastw               9    4    445    238   34.8%  ADBench-sim
  Cardio               21   10   1656    175    9.6%  ADBench-sim
  Glass                 9    4    205      9    4.2%  ADBench-sim
  Ionosphere           33   16    225    126   35.9%  ADBen

## Cell 5 — Run experiments

> **Expected runtime:**
> - 10 seeds full: ~60–120 min
> - 3 seeds quick: ~20–40 min

Each dataset row shows: gate selectivity %, MELU AUROC, ELU AUROC, Δ.

In [5]:
results  = {}   # {name: {act: [auroc per seed]}}
gate_log = {}   # {name: [gate_pct per seed]}
meta_log = {}   # {name: (dim, src, n_in, n_out, lat)}

t_total = time.time()

for nm, Xi_pool, Xo_pool, dim, src in DATASETS:
    results[nm]  = {act: [] for act in ACTS}
    gate_log[nm] = []
    n_in=len(Xi_pool); n_out=len(Xo_pool); lat=lat_for(dim)
    meta_log[nm] = (dim, src, n_in, n_out, lat)

    # Fit scaler on ALL inliers once
    sc = StandardScaler().fit(Xi_pool)
    # Scale ALL inliers once — passed to MCD so it sees the same
    # distribution the encoder was trained on (scaled data)
    Xi_pool_sc = sc.transform(Xi_pool)

    t0 = time.time()

    for seed in range(N_SEEDS):
        # ── Per-seed train/test split ──────────────────────────────────────────
        rng     = np.random.RandomState(seed)
        idx_all = rng.permutation(n_in)
        n_train = max(8, int(n_in * TRAIN_FRAC))
        Xi_tr   = sc.transform(Xi_pool[idx_all[:n_train]])   # train inliers
        Xi_te   = sc.transform(Xi_pool[idx_all[n_train:]])   # test  inliers
        Xo_te   = sc.transform(Xo_pool)                      # ALL outliers → test

        X_test  = np.vstack([Xi_te, Xo_te])
        y_test  = np.array([0]*len(Xi_te) + [1]*len(Xo_te))

        # 2stage-MELU
        try:
            au, _, gp, _ = train_two_stage(Xi_tr, Xi_pool_sc, X_test, y_test, seed=seed)
            results[nm]["2stage-MELU"].append(au)
            gate_log[nm].append(gp)
        except Exception as e:
            results[nm]["2stage-MELU"].append(0.5)

        # Baselines
        for act, af in BL_FNS.items():
            try:
                au2, _ = train_baseline(Xi_tr, X_test, y_test, af, seed=seed)
                results[nm][act].append(au2)
            except Exception:
                results[nm][act].append(0.5)

    elapsed  = time.time() - t0
    melu_mu  = np.mean(results[nm]["2stage-MELU"])
    elu_mu   = np.mean(results[nm]["ELU"])
    best     = max(np.mean(results[nm][a]) for a in ACTS)
    gp_mean  = np.mean(gate_log[nm]) if gate_log[nm] else 0.0
    gp_ok    = "✓" if 0.20 < gp_mean < 0.45 else "!"
    flag     = "★" if melu_mu >= best - 0.001 else " "

    print(f"{flag} {nm:<18} dim={dim:>3} lat={lat:>2} "
          f"gate={gp_mean:.0%}{gp_ok} "
          f"MELU={melu_mu:.4f} ELU={elu_mu:.4f} "
          f"Δ={melu_mu-elu_mu:>+.4f}  [{elapsed:.0f}s]")

print(f"\nTotal runtime: {(time.time()-t_total)/60:.1f} min")


  Wine               dim= 13 lat= 6 gate=32%✓ MELU=0.8999 ELU=0.8989 Δ=+0.0010  [3s]
  BreastCancer       dim= 30 lat=15 gate=30%✓ MELU=0.9387 ELU=0.9398 Δ=-0.0011  [19s]
  D1v7               dim= 64 lat=16 gate=29%✓ MELU=0.9799 ELU=0.9843 Δ=-0.0044  [22s]
  D3v5               dim= 64 lat=16 gate=28%✓ MELU=0.9509 ELU=0.9562 Δ=-0.0054  [22s]
★ D3v8               dim= 64 lat=16 gate=28%✓ MELU=0.9163 ELU=0.9155 Δ=+0.0008  [25s]
  D4v9               dim= 64 lat=16 gate=26%✓ MELU=0.9870 ELU=0.9889 Δ=-0.0019  [26s]
  D2v7               dim= 64 lat=16 gate=26%✓ MELU=0.9916 ELU=0.9934 Δ=-0.0018  [24s]
  Annthyroid         dim=  6 lat= 4 gate=42%✓ MELU=0.9495 ELU=0.9557 Δ=-0.0062  [184s]
★ Arrhythmia         dim=274 lat=16 gate=44%✓ MELU=1.0000 ELU=1.0000 Δ=+0.0000  [186s]
★ Breastw            dim=  9 lat= 4 gate=40%✓ MELU=0.9842 ELU=0.9846 Δ=-0.0004  [14s]
★ Cardio             dim= 21 lat=10 gate=45%✓ MELU=0.9988 ELU=0.9991 Δ=-0.0004  [59s]
★ Glass              dim=  9 lat= 4 gate=42%✓ MELU=0.

## Cell 6 — Summary table

In [6]:
DS = [d[0] for d in DATASETS]
A  = {act: np.array([np.mean(results[ds][act]) for ds in DS]) for act in ACTS}
dm = A["2stage-MELU"]

print("FINAL RESULTS TABLE")
print(f"{'Dataset':<18} {'dim':>4} {'src':>12}  "+" ".join(f"{a[:7]:>8}" for a in ACTS)+"  winner")
print("-"*90)

wins=0
for nm in DS:
    dim,src,ni,no,lat = meta_log[nm]
    vals = {a: np.mean(results[nm][a]) for a in ACTS}
    stds = {a: np.std( results[nm][a]) for a in ACTS}
    best = max(vals.values())
    row  = f"  {nm:<18} {dim:>4} {src:>12}  "
    for act in ACTS:
        v=vals[act]
        f="★" if v>=best-0.001 else " "
        row += f"{f}{v:.4f}   "
    best_act=max(vals,key=vals.get)
    row += f" {best_act[:6]}"
    print(row)
    if vals["2stage-MELU"]>=best-0.001: wins+=1

print("-"*90)
print(f"{'Overall avg':<18}{'':>4} {'':>12}  "+" ".join(f"  {A[a].mean():.4f}  " for a in ACTS))
print()
print(f"MELU-Δt wins/ties: {wins}/{len(DS)}")
print(f"Avg Δ vs ELU:   {(A['2stage-MELU']-A['ELU']).mean():>+.4f}")
print(f"Avg Δ vs Swish: {(A['2stage-MELU']-A['Swish']).mean():>+.4f}")


FINAL RESULTS TABLE
Dataset             dim          src   2stage-      ELU    Swish     GELU     ReLU  winner
------------------------------------------------------------------------------------------
  Wine                 13      sklearn   0.8999    0.8989   ★0.9066    0.9021    0.8979    Swish
  BreastCancer         30      sklearn   0.9387    0.9398   ★0.9417    0.9390    0.9370    Swish
  D1v7                 64      sklearn   0.9799   ★0.9843   ★0.9841   ★0.9840   ★0.9840    ELU
  D3v5                 64      sklearn   0.9509   ★0.9562   ★0.9556   ★0.9558   ★0.9561    ELU
  D3v8                 64      sklearn  ★0.9163   ★0.9155   ★0.9158   ★0.9154    0.9146    2stage
  D4v9                 64      sklearn   0.9870   ★0.9889   ★0.9893   ★0.9891   ★0.9893    Swish
  D2v7                 64      sklearn   0.9916   ★0.9934   ★0.9934   ★0.9934   ★0.9934    ELU
  Annthyroid            6  ADBench-sim   0.9495    0.9557   ★0.9580   ★0.9572   ★0.9578    Swish
  Arrhythmia          274  

## Cell 7 — Statistical significance tests

In [7]:
DS  = [d[0] for d in DATASETS]
A   = {act: np.array([np.mean(results[ds][act]) for ds in DS]) for act in ACTS}
dm  = A["2stage-MELU"]
bls = [a for a in ACTS if a != "2stage-MELU"]

print("="*65)
print(f"STATISTICAL TESTS  |  n={len(DS)} datasets  |  {N_SEEDS} seeds each")
print(f"Protocol: {TRAIN_FRAC:.0%}/{1-TRAIN_FRAC:.0%} inlier split per seed")
print("="*65)
print(f"\n2stage-MELU overall mean AUROC: {dm.mean():.5f}\n")

# Wilcoxon
print("Wilcoxon Signed-Rank (two-sided):")
print(f"{'Baseline':<14} {'Δ':>8} {'W':>7} {'p':>9}  Significant?")
print("-"*50)
wil = {}
for bl in bls:
    try: Wv, p = wilcoxon(dm, A[bl], alternative="two-sided")
    except: Wv, p = 0., 1.0
    sig = "✓ YES (p<0.05)" if p<0.05 else ("~ marginal (p<0.10)" if p<0.10 else "no")
    wil[bl] = dict(p=p, delta=(dm-A[bl]).mean())
    print(f"{bl:<14} {(dm-A[bl]).mean():>+8.4f} {Wv:>7.1f} {p:>9.5f}  {sig}")

# Friedman
sm  = np.column_stack([A[a] for a in ACTS])
fs, fp = friedmanchisquare(*sm.T)
rk  = np.array([rankdata(-sm[i]) for i in range(len(DS))]).mean(0)

# Critical Difference (Demšar 2006, Table 5)
k = len(ACTS); nd = len(DS)
q_table = {5:2.728,10:3.164,15:3.391,20:3.578,25:3.728,30:3.720,33:3.748}
q  = max(v for kk,v in q_table.items() if kk<=nd)
CD = q * np.sqrt(k*(k+1)/(6*nd))

wins = sum(1 for ds in DS
           if np.mean(results[ds]["2stage-MELU"]) >=
              max(np.mean(results[ds][a]) for a in ACTS) - 0.001)

print(f"\nFriedman: χ²={fs:.3f}  p={fp:.5f}  "
      f"{'SIGNIFICANT ✓' if fp<0.05 else 'not significant'}")
print(f"CD = {CD:.3f}  (k={k}, n={nd}, α=0.05)")
print(f"\nAverage Friedman ranks (lower = better):")
for act, r in sorted(zip(ACTS, rk), key=lambda x: x[1]):
    bar="█"*max(1,int((k-r+1)*5)); winner="← best" if r==min(rk) else ""
    print(f"  {act:<16}  {r:.3f}  {bar}  {winner}")
print(f"\nWins/ties: {wins}/{nd}")

# Pairwise sig
print("\nSignificant pairwise differences (|Δrank| > CD):")
from itertools import combinations
found=False
for (a1,r1),(a2,r2) in combinations(zip(ACTS,rk),2):
    if abs(r1-r2)>CD:
        print(f"  {a1} vs {a2}: |Δrank|={abs(r1-r2):.3f} > CD={CD:.3f} ✓")
        found=True
if not found: print("  None — differences not significant at α=0.05")


STATISTICAL TESTS  |  n=33 datasets  |  10 seeds each
Protocol: 50%/50% inlier split per seed

2stage-MELU overall mean AUROC: 0.98098

Wilcoxon Signed-Rank (two-sided):
Baseline              Δ       W         p  Significant?
--------------------------------------------------
ELU             -0.0015    50.0   0.00049  ✓ YES (p<0.05)
Swish           -0.0021    34.0   0.00012  ✓ YES (p<0.05)
GELU            -0.0017    36.0   0.00014  ✓ YES (p<0.05)
ReLU            -0.0015    65.0   0.00168  ✓ YES (p<0.05)

Friedman: χ²=38.400  p=0.00000  SIGNIFICANT ✓
CD = 1.459  (k=5, n=33, α=0.05)

Average Friedman ranks (lower = better):
  Swish             2.182  ███████████████████  ← best
  ELU               2.606  ████████████████  
  ReLU              2.909  ███████████████  
  GELU              3.061  ██████████████  
  2stage-MELU       4.242  ████████  

Wins/ties: 17/33

Significant pairwise differences (|Δrank| > CD):
  2stage-MELU vs ELU: |Δrank|=1.636 > CD=1.459 ✓
  2stage-MELU vs Swish: |

## Cell 8 — Figure 1: Full AUROC bar chart (all 33 datasets)

In [ ]:
DS    = [d[0] for d in DATASETS]
A     = {act: np.array([np.mean(results[ds][act]) for ds in DS]) for act in ACTS}
n_ds  = len(DS)
n_real= sum(1 for d in DATASETS if d[4]=="sklearn")

fig, ax = plt.subplots(figsize=(max(20, n_ds*0.6), 8))
x   = np.arange(n_ds); w = 0.14; offs = np.linspace(-2, 2, len(ACTS))

for i, act in enumerate(ACTS):
    means = [np.mean(results[ds][act]) for ds in DS]
    stds  = [np.std( results[ds][act]) for ds in DS]
    ax.bar(x+offs[i]*w, means, width=w,
           color=COLORS[act], alpha=0.88, label=act,
           linewidth=2.0 if act=="2stage-MELU" else 0.5,
           edgecolor="#085041" if act=="2stage-MELU" else "none")
    ax.errorbar(x+offs[i]*w, means, yerr=stds, fmt="none",
                ecolor="black", capsize=2, lw=0.7)

ax.set_xticks(x)
ax.set_xticklabels(DS, fontsize=8, rotation=45, ha='right')
ax.set_ylabel("AUROC  (reconstruction error only)", fontsize=12)
ax.set_ylim(0.3, 1.25)
ax.set_title(
    f"Two-Stage MELU-Δt vs Baselines  |  {n_ds} datasets  |  {N_SEEDS} seeds  |  "
    f"{TRAIN_FRAC:.0%}/{1-TRAIN_FRAC:.0%} train/test split per seed",
    fontsize=10)
ax.legend(fontsize=9, ncol=5)
ax.grid(axis="y", alpha=0.25)

# ★ mark winners
for xi, ds in enumerate(DS):
    best = max(np.mean(results[ds][a]) for a in ACTS)
    if np.mean(results[ds]["2stage-MELU"]) >= best - 0.001:
        ax.text(xi, 1.19, "★", ha="center", fontsize=10, color="#1D9E75")

# sklearn / ADBench-sim divider
if n_real > 0 and n_real < n_ds:
    ax.axvline(n_real-0.5, color="gray", lw=1.5, ls="--", alpha=0.5)
    ax.text(n_real*0.5-0.5,  1.22, "sklearn (real)", ha="center", fontsize=8, color="gray")
    ax.text(n_real+(n_ds-n_real)*0.5-0.5, 1.22, "ADBench-sim",
            ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig("outputs/fig1_auroc_all.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/fig1_auroc_all.png")


## Cell 9 — Figure 2: Statistical analysis (delta + CD + Wilcoxon + ranks)

In [ ]:
DS  = [d[0] for d in DATASETS]
A   = {act: np.array([np.mean(results[ds][act]) for ds in DS]) for act in ACTS}
dm  = A["2stage-MELU"]
bls = [a for a in ACTS if a != "2stage-MELU"]
sm  = np.column_stack([A[a] for a in ACTS])
_,fp= friedmanchisquare(*sm.T)
rk  = np.array([rankdata(-sm[i]) for i in range(len(DS))]).mean(0)
k   = len(ACTS); nd = len(DS)
q   = max(v for kk,v in {5:2.728,10:3.164,20:3.578,30:3.720,33:3.748}.items() if kk<=nd)
CD  = q*np.sqrt(k*(k+1)/(6*nd))
wil2= {}
for bl in bls:
    try: _, p=wilcoxon(dm,A[bl],alternative="two-sided")
    except: p=1.0
    wil2[bl]=dict(p=p, delta=(dm-A[bl]).mean())

n_real=sum(1 for d in DATASETS if d[4]=="sklearn")
fig   =plt.figure(figsize=(22,16))
gs    =gridspec.GridSpec(3,3,figure=fig,hspace=0.48,wspace=0.36)
fig.suptitle("Two-Stage MELU-Δt: Statistical Analysis
"
             f"{nd} datasets  |  {N_SEEDS} seeds  |  "
             f"{TRAIN_FRAC:.0%}/{1-TRAIN_FRAC:.0%} inlier train/test split per seed",
             fontsize=12)

# ── Panel 1: Delta vs ELU ──────────────────────────────────────────────────────
ax=fig.add_subplot(gs[0,:])
d_e=[np.mean(results[ds]["2stage-MELU"])-np.mean(results[ds]["ELU"])   for ds in DS]
d_s=[np.mean(results[ds]["2stage-MELU"])-np.mean(results[ds]["Swish"]) for ds in DS]
x2=np.arange(nd)
ax.bar(x2-0.20,d_e,width=0.38,color="#1D9E75",alpha=0.80,label="vs ELU")
ax.bar(x2+0.20,d_s,width=0.38,color="#534AB7",alpha=0.80,label="vs Swish")
ax.axhline(0,  color="black",lw=0.8)
ax.axhline(0.01, color="#1D9E75",lw=1,ls="--",alpha=0.4)
ax.axhline(-0.01,color="#D85A30", lw=1,ls="--",alpha=0.4)
ax.fill_between([-0.5,nd-0.5],[0.01]*2,[0.2]*2,alpha=0.04,color="#1D9E75")
ax.fill_between([-0.5,nd-0.5],[-0.15]*2,[-0.01]*2,alpha=0.04,color="#D85A30")
ax.set_xticks(x2); ax.set_xticklabels(DS,fontsize=7,rotation=45,ha='right')
ax.set_ylabel("AUROC (2stage-MELU − baseline)",fontsize=11)
ax.set_title("Per-dataset advantage over strongest baselines (positive = MELU wins)",fontsize=10)
ax.legend(fontsize=9); ax.grid(axis="y",alpha=0.25)
if n_real>0 and n_real<nd:
    ax.axvline(n_real-0.5,color="gray",lw=1.5,ls="--",alpha=0.5)

# ── Panel 2: Gate selectivity ──────────────────────────────────────────────────
ax=fig.add_subplot(gs[1,0])
gate_means=[np.mean(gate_log[ds])*100 if gate_log.get(ds) else 0.0 for ds in DS]
colors_g=["#1D9E75" if 20<g<45 else "#D85A30" for g in gate_means]
ax.barh(DS,gate_means,color=colors_g,alpha=0.80)
ax.axvline(25,color="#1D9E75",lw=1.5,ls="--",alpha=0.7,label="25% lower")
ax.axvline(40,color="#D85A30",lw=1.5,ls="--",alpha=0.7,label="40% upper")
ax.set_xlabel("Gate fires on % of inliers")
ax.set_title("Gate selectivity per dataset
(target: 25–40% → MCD reliable)",fontsize=10)
ax.tick_params(labelsize=7); ax.legend(fontsize=8)

# ── Panel 3: Wilcoxon p-values ─────────────────────────────────────────────────
ax=fig.add_subplot(gs[1,1])
pvals=[wil2[b]["p"] for b in bls]
cols =["#1D9E75" if p<0.05 else "#BA7517" if p<0.10 else "#D85A30" for p in pvals]
ax.barh(bls,pvals,color=cols,alpha=0.85)
ax.axvline(0.05,color="black",lw=1.5,ls="--",label="α=0.05")
ax.axvline(0.10,color="gray", lw=1.0,ls=":",  label="α=0.10")
ax.set_xlabel("p-value (lower = more significant)")
ax.set_title("Wilcoxon Signed-Rank
(MELU-Δt vs each baseline)",fontsize=10)
ax.legend(fontsize=8)
for i,(b,p) in enumerate(zip(bls,pvals)):
    ax.text(p+0.005,i,f"{p:.4f}",va="center",fontsize=9)

# ── Panel 4: Friedman ranks ────────────────────────────────────────────────────
ax=fig.add_subplot(gs[1,2])
sp=sorted(zip(ACTS,rk),key=lambda x:x[1])
ax.barh([x[0] for x in sp],[x[1] for x in sp],
        color=[COLORS[x[0]] for x in sp],alpha=0.85)
ax.axvline(min(rk)+CD,color="gray",lw=1.5,ls=":",label=f"CD={CD:.2f}")
ax.set_xlabel("Average Friedman rank (lower = better)")
ax.set_title("Friedman average ranks",fontsize=10)
ax.legend(fontsize=8)
for nm_,r_ in sp: ax.text(r_+0.02,nm_,f"{r_:.2f}",va="center",fontsize=9)

# ── Panel 5: Critical Difference diagram ──────────────────────────────────────
ax=fig.add_subplot(gs[2,:])
ax.axis("off")
ax.set_xlim(0.3,k+0.7); ax.set_ylim(-1.5,4.5); ax.invert_xaxis()
ax.axhline(0,color="black",lw=2.5,xmin=0.02,xmax=0.98)
for i in range(1,k+1):
    ax.plot(i,0,"k|",ms=14,mew=2.5)
    ax.text(i,-0.50,str(i),ha="center",fontsize=13)
ax.text((k+1)/2,-1.10,"← Average rank (lower = better)",
        ha="center",fontsize=11,color="gray")

sp2=sorted(zip(ACTS,rk),key=lambda x:x[1])
for i,(nm_,r_) in enumerate(sp2):
    yp=1.6 if i%2==0 else 2.6; c=COLORS[nm_]; lw=3.0 if nm_=="2stage-MELU" else 1.8
    ax.plot([r_,r_],[0,yp],color=c,lw=lw,ls="--",alpha=0.85)
    ax.plot(r_,yp,"o",color=c,ms=14,zorder=5,
            markeredgecolor="#085041" if nm_=="2stage-MELU" else "none",
            markeredgewidth=2.0)
    ax.text(r_,yp+0.36,nm_,ha="center",fontsize=10,
            fontweight="bold" if nm_=="2stage-MELU" else "normal",color=c)
    ax.text(r_,yp-0.34,f"{r_:.2f}",ha="center",fontsize=9,color="gray")

br=min(rk)
ax.annotate("",xy=(br+CD,3.8),xytext=(br,3.8),
            arrowprops=dict(arrowstyle="<->",color="black",lw=2.5))
ax.text(br+CD/2,4.05,f"CD = {CD:.3f}",ha="center",fontsize=13,fontweight="bold")
ax.set_title(
    f"Critical Difference Diagram  |  n={nd} datasets, k={k}, α=0.05  |  "
    f"Friedman p={fp:.4f}  "
    f"({'Significant ✓' if fp<0.05 else 'Not significant — differences within CD'})",
    fontsize=10,pad=8)

plt.savefig("outputs/fig2_stats.png",dpi=150,bbox_inches="tight")
plt.show()
print("Saved → outputs/fig2_stats.png")


## Cell 10 — Figure 3: Real datasets separate (sklearn only, clean comparison)

In [ ]:
# Only real sklearn datasets — no simulation noise
real_ds=[nm for nm,_,_,_,src in DATASETS if src=="sklearn"]
A_real ={act:np.array([np.mean(results[ds][act]) for ds in real_ds]) for act in ACTS}
dm_r   =A_real["2stage-MELU"]
nd_r   =len(real_ds)

fig,axes=plt.subplots(1,2,figsize=(16,6))
fig.suptitle(f"Real sklearn datasets only  |  {nd_r} datasets  |  {N_SEEDS} seeds  |  "
             f"{TRAIN_FRAC:.0%}/{1-TRAIN_FRAC:.0%} split",fontsize=11)

# Bar chart
ax=axes[0]; x=np.arange(nd_r); w=0.14; offs=np.linspace(-2,2,len(ACTS))
for i,act in enumerate(ACTS):
    means=[np.mean(results[ds][act]) for ds in real_ds]
    stds =[np.std( results[ds][act]) for ds in real_ds]
    ax.bar(x+offs[i]*w,means,width=w,color=COLORS[act],alpha=0.88,label=act,
           linewidth=2.0 if act=="2stage-MELU" else 0.5,
           edgecolor="#085041" if act=="2stage-MELU" else "none")
    ax.errorbar(x+offs[i]*w,means,yerr=stds,fmt="none",ecolor="black",capsize=3,lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(real_ds,fontsize=10)
ax.set_ylabel("AUROC (recon error only)",fontsize=11); ax.set_ylim(0.5,1.15)
ax.set_title("AUROC per dataset (10 seeds ±std)",fontsize=10)
ax.legend(fontsize=9,ncol=3); ax.grid(axis="y",alpha=0.25)
for xi,ds in enumerate(real_ds):
    best=max(np.mean(results[ds][a]) for a in ACTS)
    if np.mean(results[ds]["2stage-MELU"])>=best-0.001:
        ax.text(xi,1.10,"★",ha="center",fontsize=12,color="#1D9E75")

# Delta per dataset
ax=axes[1]
d_e=[np.mean(results[ds]["2stage-MELU"])-np.mean(results[ds]["ELU"])   for ds in real_ds]
d_s=[np.mean(results[ds]["2stage-MELU"])-np.mean(results[ds]["Swish"]) for ds in real_ds]
x2=np.arange(nd_r)
bars_e=ax.bar(x2-0.18,d_e,width=0.34,color="#1D9E75",alpha=0.85,label="vs ELU")
bars_s=ax.bar(x2+0.18,d_s,width=0.34,color="#534AB7",alpha=0.85,label="vs Swish")
ax.axhline(0,color="black",lw=0.8)
ax.axhline(0.005,color="#1D9E75",lw=1,ls="--",alpha=0.4,label="Δ=+0.005")
for b,v in zip(bars_e,d_e):
    ax.text(b.get_x()+b.get_width()/2,v+(0.001 if v>=0 else -0.0015),
            f"{v:+.3f}",ha="center",fontsize=9,fontweight="bold",
            color="#085041" if v>=0 else "#993C1D")
ax.set_xticks(x2); ax.set_xticklabels(real_ds,fontsize=10)
ax.set_ylabel("AUROC (2stage-MELU − baseline)",fontsize=11)
ax.set_title("Advantage over baselines per dataset",fontsize=10)
ax.legend(fontsize=9); ax.grid(axis="y",alpha=0.25)

# Wilcoxon on real datasets only
print("Wilcoxon (real sklearn datasets only):")
for bl in [a for a in ACTS if a!="2stage-MELU"]:
    try: _,p=wilcoxon(dm_r,A_real[bl],alternative="two-sided")
    except: p=1.0
    sig="✓" if p<0.05 else "~" if p<0.10 else "no"
    print(f"  vs {bl:<14} Δ={( dm_r-A_real[bl]).mean():>+.4f}  p={p:.4f}  {sig}")

plt.tight_layout()
plt.savefig("outputs/fig3_real_datasets.png",dpi=150,bbox_inches="tight")
plt.show()
print("\nSaved → outputs/fig3_real_datasets.png")


## Cell 11 — Export results to CSV

In [ ]:
DS=[d[0] for d in DATASETS]
A ={act:np.array([np.mean(results[ds][act]) for ds in DS]) for act in ACTS}
dm=A["2stage-MELU"]

# Full results
rows=[]
for nm in DS:
    dim,src,ni,no,lat=meta_log[nm]
    gp=np.mean(gate_log[nm]) if gate_log.get(nm) else float("nan")
    for act in ACTS:
        rows.append({
            "dataset":nm, "source":src, "dim":dim, "lat":lat,
            "n_inlier_pool":ni, "n_outlier_pool":no,
            "train_frac":TRAIN_FRAC, "n_seeds":N_SEEDS,
            "score_mode":"recon_only", "loss":"MAE+BCE",
            "validation":"per_seed_50_50_split",
            "activation":act,
            "auroc_mean":round(np.mean(results[nm][act]),4),
            "auroc_std": round(np.std( results[nm][act]),4),
            "gate_pct":round(gp*100,1) if act=="2stage-MELU" else "",
        })
pd.DataFrame(rows).to_csv("outputs/melu_benchmark_full.csv",index=False)

# Summary
srows=[]
for nm in DS:
    dim,src,ni,no,lat=meta_log[nm]
    gp=np.mean(gate_log[nm]) if gate_log.get(nm) else float("nan")
    vals={a:round(np.mean(results[nm][a]),4) for a in ACTS}
    best=max(vals.values())
    srows.append({
        "dataset":nm,"source":src,"dim":dim,"n_in":ni,"n_out":no,
        "gate_pct":round(gp*100,1),
        **vals,
        "melu_wins":vals["2stage-MELU"]>=best-0.001,
        "delta_vs_ELU":  round(vals["2stage-MELU"]-vals["ELU"],4),
        "delta_vs_Swish":round(vals["2stage-MELU"]-vals["Swish"],4),
    })
pd.DataFrame(srows).to_csv("outputs/melu_benchmark_summary.csv",index=False)

# Stats
sm  = np.column_stack([A[a] for a in ACTS])
fs, fp2 = friedmanchisquare(*sm.T)
rk2 = np.array([rankdata(-sm[i]) for i in range(len(DS))]).mean(0)
nd  = len(DS); k=len(ACTS)
q   = max(v for kk,v in {5:2.728,10:3.164,20:3.578,33:3.748}.items() if kk<=nd)
CD2 = q*np.sqrt(k*(k+1)/(6*nd))
wins2=sum(1 for ds in DS if np.mean(results[ds]["2stage-MELU"])>=
          max(np.mean(results[ds][a]) for a in ACTS)-0.001)

stat_rows=[
    {"metric":"n_datasets",      "value":nd},
    {"metric":"n_seeds",         "value":N_SEEDS},
    {"metric":"train_frac",      "value":TRAIN_FRAC},
    {"metric":"validation",      "value":"per_seed_50_50_inlier_split"},
    {"metric":"melu_wins",       "value":wins2},
    {"metric":"melu_mean_auroc", "value":round(float(dm.mean()),5)},
    {"metric":"friedman_chi2",   "value":round(float(fs),3)},
    {"metric":"friedman_p",      "value":round(fp2,5)},
    {"metric":"CD",              "value":round(CD2,3)},
    {"metric":"melu_rank",       "value":round(float(rk2[0]),3)},
]
for bl in [a for a in ACTS if a!="2stage-MELU"]:
    try: _,p=wilcoxon(dm,A[bl],alternative="two-sided")
    except: p=1.0
    stat_rows.append({"metric":f"wilcoxon_p_vs_{bl}","value":round(p,5)})
    stat_rows.append({"metric":f"delta_vs_{bl}","value":round(float((dm-A[bl]).mean()),5)})

pd.DataFrame(stat_rows).to_csv("outputs/melu_benchmark_stats.csv",index=False)

print("Saved:")
print("  outputs/melu_benchmark_full.csv     (all seeds, all datasets, all activations)")
print("  outputs/melu_benchmark_summary.csv  (one row per dataset)")
print("  outputs/melu_benchmark_stats.csv    (Friedman, Wilcoxon, CD, ranks)")
print()
print(f"2stage-MELU mean:  {dm.mean():.4f}")
print(f"ELU mean:          {A['ELU'].mean():.4f}")
print(f"Δ(MELU - ELU):     {(dm-A['ELU']).mean():>+.4f}")
print(f"Wins/ties:         {wins2}/{nd}")
print(f"Friedman p={fp2:.5f}  CD={CD2:.3f}  MELU rank={rk2[0]:.2f}")
